# CONSORT paragraph section labeling — global article input

This notebook implements the new first-stage logic:

1. Use the whole article as global context.
2. Produce paragraph-level section labels with two fields:
   - `location_section`: where the paragraph appears structurally.
   - `content_sections`: what the paragraph is actually talking about; this can contain multiple sections.
3. Save compact JSON keyed by article and paragraph number.
4. Evaluate the section labels against the original dataset section labels.

The downstream customized extraction modules and GEPA reflection are intentionally left for the next notebook/version.

In [1]:
import os
import re
import json
import ast
from pathlib import Path
from collections import Counter, defaultdict

import numpy as np
import pandas as pd
from tqdm.auto import tqdm

try:
    import dspy
except ImportError:
    dspy = None
    print("DSPy is not installed in this environment. Install/configure DSPy before running LLM section labeling.")

## 1. Configuration

The LLM input should not include the original dataset section name. The dataset section is retained only as `gold_location_section` / `original_location_section` for evaluation. Set `USE_DATASET_LOCATION_AS_TYPE1=False` when you want the LLM to infer both structural location and content-based section labels from paragraph text and article order.


In [2]:
# -----------------------------
# Data paths
# -----------------------------
DATA_PATH = Path("../data/paragraphs_with_sections.csv")
SENTENCE_PATH = Path("../data/sentences.csv")  # optional; only used if available for downstream alignment/debugging
OUTPUT_DIR = Path("../output_and_result/section_labeling")
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

SECTION_JSON_PATH = OUTPUT_DIR / "paragraph_section_labels_global_v1.json"
SECTION_CSV_PATH = OUTPUT_DIR / "paragraph_section_labels_global_v1.csv"
EVAL_CSV_PATH = OUTPUT_DIR / "section_label_eval_v1.csv"

# -----------------------------
# Runtime options
# -----------------------------
ARTICLE_ID_COL = "article"
PARAGRAPH_ID_COL = "pid"
PARAGRAPH_TEXT_COL = "text"
ORIGINAL_SECTION_COL = "section"

# For the current dataset, the original section column is the location-based section.
USE_DATASET_LOCATION_AS_TYPE1 = False

# Run on a small subset first while debugging; set to None for all articles.
MAX_ARTICLES = 5

# Long articles can exceed the context window. The labeler uses compact paragraph text, but this is still useful.
MAX_CHARS_PER_PARAGRAPH = 1200
MAX_ARTICLE_CHARS = 90000

RANDOM_SEED = 42

In [7]:
CANONICAL_SECTIONS = [
    "title_abstract",
    "introduction",
    "methods",
    "results",
    "discussion",
    "other_information",
]

SECTION_DESCRIPTION = """
Allowed canonical sections:
- title_abstract: article title, abstract, summary, trial registry summary if it functions like an abstract
- introduction: background, rationale, objectives, hypotheses before methods
- methods: trial design, participants, interventions, outcomes, randomization, blinding, sample size, statistical methods, protocol details
- results: participant flow, recruitment dates, baseline data, numbers analyzed, outcome results, ancillary analyses, harms/adverse events
- discussion: limitations, generalizability, interpretation, implications, conclusions
- other_information: registration, protocol availability, funding, conflicts of interest, acknowledgements, author contributions, ethics statements, data availability, references, supplements, tables/figures captions when not otherwise classifiable
""".strip()

## 2. Load and normalize the paragraph dataset

In [8]:
def require_columns(df, required, name="dataframe"):
    missing = [c for c in required if c not in df.columns]
    if missing:
        raise ValueError(f"{name} is missing required columns: {missing}. Available columns: {list(df.columns)}")


def load_paragraph_dataset(path=DATA_PATH):
    df = pd.read_csv(path)
    require_columns(df, [ARTICLE_ID_COL, PARAGRAPH_ID_COL, PARAGRAPH_TEXT_COL, ORIGINAL_SECTION_COL], "paragraph dataset")
    df = df.copy()
    df[PARAGRAPH_TEXT_COL] = df[PARAGRAPH_TEXT_COL].fillna("").astype(str)
    df[ORIGINAL_SECTION_COL] = df[ORIGINAL_SECTION_COL].fillna("Other").astype(str)
    # Stable within-article paragraph numbering used in the JSON output to save tokens.
    df = df.sort_values([ARTICLE_ID_COL, PARAGRAPH_ID_COL]).reset_index(drop=True)
    df["pnum"] = df.groupby(ARTICLE_ID_COL).cumcount()
    return df

paragraph_df = load_paragraph_dataset()
print(paragraph_df.shape)
paragraph_df.head()

(4124, 7)


,article,pid,text,charStartAt,charEndAt,section,pnum
0,PMC3002766,1,Title,0,5,Title,0
1,PMC3002766,2,Bosentan treatment of digital ulcers related t...,6,150,Title,1
2,PMC3002766,3,Abstract,151,159,Abstract,2
3,PMC3002766,4,Objectives,180,190,Abstract,3
4,PMC3002766,5,Ischaemic digital ulcers (DUs) are common in p...,201,747,Abstract,4


## 3. Section normalization helpers

The dataset may contain section names such as `Abstract`, `Methods`, `Results`, or more granular headings. These helpers map them into the six-section schema agreed on for the new design.

In [9]:
def normalize_section_name(raw):
    """Map raw section headings to the six canonical section labels."""
    s = str(raw or "").strip().lower()
    s = re.sub(r"[_\-]+", " ", s)
    s = re.sub(r"\s+", " ", s)

    if not s:
        return "other_information"

    # Title / abstract
    if any(k in s for k in ["title", "abstract", "summary", "synopsis"]):
        return "title_abstract"

    # Introduction
    if any(k in s for k in ["introduction", "background", "rationale", "objective", "objectives"]):
        return "introduction"

    # Methods
    method_terms = [
        "method", "methods", "material", "materials", "design", "participant", "patients",
        "intervention", "outcome", "random", "randomisation", "randomization", "blinding",
        "masking", "sample size", "statistical", "statistics", "analysis", "protocol",
        "eligibility", "recruitment", "allocation", "procedure"
    ]
    if any(k in s for k in method_terms):
        return "methods"

    # Results
    result_terms = [
        "result", "results", "finding", "findings", "participant flow", "baseline",
        "numbers analysed", "numbers analyzed", "outcome", "estimation", "harms",
        "adverse", "recruitment", "ancillary", "subgroup"
    ]
    if any(k in s for k in result_terms):
        return "results"

    # Discussion
    discussion_terms = [
        "discussion", "limitation", "limitations", "generalizability", "generalisability",
        "interpretation", "conclusion", "conclusions", "implication", "implications"
    ]
    if any(k in s for k in discussion_terms):
        return "discussion"

    # Admin/other information
    other_terms = [
        "funding", "registration", "trial registration", "protocol", "conflict", "interest",
        "acknowledg", "ethic", "author", "contribution", "data availability", "supplement",
        "reference", "appendix", "figure", "table"
    ]
    if any(k in s for k in other_terms):
        return "other_information"

    return "other_information"

paragraph_df["original_location_section"] = paragraph_df[ORIGINAL_SECTION_COL].apply(normalize_section_name)
paragraph_df[[ARTICLE_ID_COL, PARAGRAPH_ID_COL, "pnum", ORIGINAL_SECTION_COL, "original_location_section"]].head(10)

,article,pid,pnum,section,original_location_section
0,PMC3002766,1,0,Title,title_abstract
1,PMC3002766,2,1,Title,title_abstract
2,PMC3002766,3,2,Abstract,title_abstract
3,PMC3002766,4,3,Abstract,title_abstract
4,PMC3002766,5,4,Abstract,title_abstract
5,PMC3002766,6,5,Abstract,title_abstract
6,PMC3002766,7,6,Abstract,title_abstract
7,PMC3002766,8,7,Abstract,title_abstract
8,PMC3002766,9,8,Abstract,title_abstract
9,PMC3002766,10,9,Abstract,title_abstract


In [10]:
print("Raw section counts:")
display(paragraph_df[ORIGINAL_SECTION_COL].value_counts().to_frame("n"))

print("\nCanonical original/location section counts:")
display(paragraph_df["original_location_section"].value_counts().reindex(CANONICAL_SECTIONS, fill_value=0).to_frame("n"))

Raw section counts:


,n
section,
Methods,1001
Results,948
Discussion,931
Abstract,515
Other,388
Introduction,241
Title,100



Canonical original/location section counts:


,n
original_location_section,
title_abstract,615
introduction,241
methods,1001
results,948
discussion,931
other_information,388


## 4. Build compact global article input

Each LLM call receives the full article in paragraph-numbered format. The output only needs to reference `pnum`, not repeat the paragraph text.

In [11]:
def compact_text(text, max_chars=MAX_CHARS_PER_PARAGRAPH):
    text = re.sub(r"\s+", " ", str(text or "")).strip()
    if len(text) <= max_chars:
        return text
    return text[:max_chars].rstrip() + " ..."


def build_article_input(article_df, include_dataset_location=False):
    rows = []
    for _, row in article_df.sort_values("pnum").iterrows():
        pnum = int(row["pnum"])
        text = compact_text(row[PARAGRAPH_TEXT_COL])
        # Do not include original/dataset section labels in the LLM input.
        # They are gold metadata used only for evaluation.
        rows.append(f"[p{pnum}] {text}")
    article_text = "\n".join(rows)
    if len(article_text) > MAX_ARTICLE_CHARS:
        article_text = article_text[:MAX_ARTICLE_CHARS].rstrip() + "\n[TRUNCATED_FOR_CONTEXT_WINDOW]"
    return article_text

sample_article = paragraph_df[ARTICLE_ID_COL].iloc[0]
sample_input = build_article_input(paragraph_df[paragraph_df[ARTICLE_ID_COL] == sample_article])
print(sample_input[:3000])


[p0] Title
[p1] Bosentan treatment of digital ulcers related to systemic sclerosis: results from the RAPIDS-2 randomised, double-blind, placebo-controlled trial
[p2] Abstract
[p3] Objectives
[p4] Ischaemic digital ulcers (DUs) are common in patients with systemic sclerosis (SSc) and are a cause of disease-related morbidity. In an earlier trial, treatment with bosentan, an oral endothelin receptor antagonist, reduced the occurrence of new DUs by 48%. The present study (RAPIDS-2, for ‘RAndomized, double-blind, Placebo-controlled study with bosentan on healing and prevention of Ischemic Digital ulcers in patients with systemic Sclerosis’) was conducted to more fully evaluate the effects of bosentan treatment on DUs associated with SSc.
[p5] Methods
[p6] This double-blind, placebo-controlled trial conducted at 41 centres in Europe and North America randomised 188 patients with SSc with at least 1 active DU (‘cardinal ulcer’) to bosentan 62.5 mg twice daily for 4 weeks and 125 mg twice dail

## 5. DSPy section labeler

The labeler returns one JSON object per article. Both `location_section` and `content_sections` are predicted from paragraph text and global article order. The original dataset section is not shown to the model and is used only for evaluation.

```json
{
  "article": "...",
  "paragraph_sections": [
    {
      "pnum": 0,
      "location_section": "title_abstract",
      "content_sections": ["title_abstract"]
    }
  ]
}
```


In [31]:
if dspy is not None:
    class ArticleSectionLabeling(dspy.Signature):
        """
        Label paragraph-level sections for a CONSORT randomized trial article using the whole article as context.

        You must use exactly these canonical labels:
        title_abstract, introduction, methods, results, discussion, other_information.

        Definitions:
        - location_section means the structural location of the paragraph in the article.
        - content_sections means the actual rhetorical/content role of the paragraph. It can contain multiple labels.

        Important rules:
        - Return valid JSON only. No markdown and no explanation.
        - Include every paragraph number present in the article input exactly once.
        - Do not repeat paragraph text in the output.
        - content_sections must be a non-empty list.
        - A paragraph under Introduction can still have content_sections=["results"] if it reports actual findings.
        - A paragraph under Methods can still include content_sections=["methods", "results"] if it mixes methods and outcome results.
        - Use other_information for registration, funding, conflicts, protocol availability, ethics, author contributions, references, or miscellaneous article metadata.
        
        Output format (use exactly this structure):
        {
        "article_id": "<article_id>",
        "paragraph_sections": [
            {"pnum": 0, "location_section": "title_abstract", "content_sections": ["title_abstract"]},
            {"pnum": 1, "location_section": "introduction", "content_sections": ["introduction", "results"]}
        ]
        }
        """

        article_id = dspy.InputField(desc="Article identifier")
        section_schema = dspy.InputField(desc="Allowed section labels and definitions")
        article_text = dspy.InputField(desc="Full article text in paragraph-numbered format")
        output_json = dspy.OutputField(desc="Valid JSON object with article and paragraph_sections")


    class ArticleSectionLabeler(dspy.Module):
        def __init__(self):
            super().__init__()
            self.label = dspy.Predict(ArticleSectionLabeling)

        def forward(self, article_id, article_text, use_dataset_location_as_type1=False):
            output = self.label(
                article_id=str(article_id),
                section_schema=SECTION_DESCRIPTION,
                article_text=article_text,
            )
            print(output)
            return self.label(
                article_id=str(article_id),
                section_schema=SECTION_DESCRIPTION,
                article_text=article_text,
            )
else:
    ArticleSectionLabeler = None

## 6. JSON parsing and validation helpers

In [33]:
def extract_json_object(text):
    """Best-effort extraction of a JSON object from an LLM response."""
    if isinstance(text, dict):
        return text
    text = str(text or "").strip()
    if not text:
        raise ValueError("Empty output")
    # Remove markdown fences if present.
    text = re.sub(r"^```(?:json)?", "", text.strip(), flags=re.IGNORECASE).strip()
    text = re.sub(r"```$", "", text.strip()).strip()
    try:
        return json.loads(text)
    except json.JSONDecodeError:
        pass
    start = text.find("{")
    end = text.rfind("}")
    if start >= 0 and end > start:
        return json.loads(text[start:end+1])
    raise ValueError("Could not find valid JSON object")


def clean_section_list(value):
    if isinstance(value, str):
        # Accept a single string or a string representation of a list.
        v = value.strip()
        try:
            parsed = ast.literal_eval(v)
            if isinstance(parsed, list):
                value = parsed
            else:
                value = [v]
        except Exception:
            value = [v]
    if not isinstance(value, list):
        value = [value]
    cleaned = []
    for x in value:
        sec = normalize_section_name(x)
        if sec in CANONICAL_SECTIONS and sec not in cleaned:
            cleaned.append(sec)
    return cleaned or ["other_information"]


def validate_and_complete_article_output(raw_obj, article_df, use_dataset_location_as_type1=False):
    """Normalize, validate, and fill missing paragraph predictions.

    Important: original_location_section is gold metadata for evaluation only.
    It is never copied into location_section unless you deliberately set
    use_dataset_location_as_type1=True for an oracle/debug run.
    """
    article_id = str(article_df[ARTICLE_ID_COL].iloc[0])
    by_pnum = {}

    for item in raw_obj.get("paragraph_sections", []):
        try:
            pnum = int(str(item.get("pnum")).replace("p", ""))
        except Exception:
            continue
        loc = normalize_section_name(item.get("location_section", "other_information"))
        cont = clean_section_list(item.get("content_sections", []))
        by_pnum[pnum] = {
            "pnum": pnum,
            "location_section": loc,
            "content_sections": cont,
            "missing_filled": False,
        }

    completed = []
    for _, row in article_df.sort_values("pnum").iterrows():
        pnum = int(row["pnum"])
        pred = by_pnum.get(pnum, None)
        original_loc = row["original_location_section"]

        if pred is None:
            # Do not silently copy the gold label for real evaluation.
            pred = {
                "pnum": pnum,
                "location_section": "other_information",
                "content_sections": ["other_information"],
                "missing_filled": True,
            }

        if use_dataset_location_as_type1:
            # Oracle/debug mode only. Do not use this for evaluating location labeling.
            pred["location_section"] = original_loc
            pred["oracle_location_copied"] = True

        pred["article"] = article_id
        pred[PARAGRAPH_ID_COL] = row[PARAGRAPH_ID_COL]
        pred["original_section_raw"] = row[ORIGINAL_SECTION_COL]
        pred["original_location_section"] = original_loc
        completed.append(pred)

    return {
        "article": article_id,
        "paragraph_sections": completed,
    }

def heuristic_article_output(article_df):
    """Non-LLM fallback: content section equals normalized original location section."""
    article_id = str(article_df[ARTICLE_ID_COL].iloc[0])
    rows = []
    for _, row in article_df.sort_values("pnum").iterrows():
        loc = row["original_location_section"]
        rows.append({
            "article": article_id,
            "pnum": int(row["pnum"]),
            PARAGRAPH_ID_COL: row[PARAGRAPH_ID_COL],
            "location_section": loc,
            "content_sections": [loc],
            "original_section_raw": row[ORIGINAL_SECTION_COL],
            "original_location_section": loc,
            "heuristic_fallback": True,
        })
    return {"article": article_id, "paragraph_sections": rows}

## 7. Run section labeling

Configure your DSPy LM before running this cell. Example:

```python
lm = dspy.LM("openai/gpt-4o-mini", api_key=os.environ["OPENAI_API_KEY"])
dspy.configure(lm=lm)
```

If DSPy is unavailable or no LM is configured, the notebook will use a deterministic fallback where `content_sections` equals the normalized original location section. That fallback is useful for testing the pipeline but not for final content-based section labels.

In [18]:
from llama_index.llms.ollama import Ollama

api_key = input("Enter your OpenAI API key: ")
# import dspy
import re as _re_for_thinking

#lm = dspy.LM("openai/gpt-5.4-mini-2026-03-17", temperature=1, api_key=api_key, max_retries=10,reasoning_effort="medium")
#lm = dspy.LM("openai/o4-mini-2025-04-16", temperature=1, api_key=api_key, max_retries=10)
#lm = dspy.LM("openai/o3-2025-04-16", temperature=1, api_key=api_key, max_retries=10)


class ThinkingModelLM(dspy.LM):
    """dspy.LM subclass that strips <think>...</think> blocks from responses.
    Use for thinking models (Qwen3, etc.) so DSPy adapters see clean output."""
    _THINK_RE = _re_for_thinking.compile(r'<think>[\s\S]*?</think>\s*', _re_for_thinking.DOTALL)

    def __call__(self, prompt=None, messages=None, **kwargs):
        outputs = super().__call__(prompt=prompt, messages=messages, **kwargs)
        return [self._strip(o) for o in outputs]

    def _strip(self, output):
        if isinstance(output, str):
            return self._THINK_RE.sub('', output).strip()
        return output


# For thinking models (Qwen3, etc.), use ThinkingModelLM:
# lm = ThinkingModelLM(
#     "openai/gemma4:31b",
#     api_base="http://localhost:11434/v1",
#     api_key="ollama",
#     temperature=1,
#     max_tokens=200000,
# )

# For cloud models (OpenAI, etc.), use dspy.LM directly:
lm = dspy.LM("openai/o4-mini-2025-04-16", temperature=1, api_key=api_key, max_retries=10)

dspy.configure(lm=lm)

In [34]:
def has_configured_dspy_lm():
    if dspy is None:
        return False
    try:
        _ = dspy.settings.lm
        return dspy.settings.lm is not None
    except Exception:
        return False


def run_article_section_labeling(paragraph_df, max_articles=MAX_ARTICLES, use_dataset_location_as_type1=USE_DATASET_LOCATION_AS_TYPE1):
    article_ids = list(paragraph_df[ARTICLE_ID_COL].drop_duplicates())
    if max_articles is not None:
        article_ids = article_ids[:max_articles]

    use_llm = has_configured_dspy_lm()
    if use_llm:
        labeler = ArticleSectionLabeler()
        print(f"Running LLM section labeling on {len(article_ids)} articles...")
    else:
        labeler = None
        print("No configured DSPy LM found. Using heuristic fallback for pipeline testing.")

    outputs = []
    errors = []
    for article_id in tqdm(article_ids):
        article_df = paragraph_df[paragraph_df[ARTICLE_ID_COL] == article_id].copy()
        try:
            if use_llm:
                article_text = build_article_input(
                    article_df,
                    include_dataset_location=False,
                )
                pred = labeler(
                    article_id=article_id,
                    article_text=article_text,
                    use_dataset_location_as_type1=use_dataset_location_as_type1,
                )
                raw_obj = extract_json_object(pred.output_json)
                clean_obj = validate_and_complete_article_output(
                    raw_obj,
                    article_df,
                    use_dataset_location_as_type1=use_dataset_location_as_type1,
                )
            else:
                clean_obj = heuristic_article_output(article_df)
            outputs.append(clean_obj)
        except Exception as e:
            errors.append({"article": article_id, "error": repr(e)})
            outputs.append(heuristic_article_output(article_df))

    return outputs, pd.DataFrame(errors)

section_outputs, section_errors = run_article_section_labeling(paragraph_df)
print(f"articles labeled: {len(section_outputs)}")
print(f"errors: {len(section_errors)}")
section_errors.head()

# Sanity checks: real LLM evaluation should have zero heuristic fallback rows and few/no missing_filled rows.
section_pred_df_tmp = outputs_to_dataframe(section_outputs) if "outputs_to_dataframe" in globals() else pd.DataFrame()
if len(section_pred_df_tmp):
    print("heuristic fallback rows:", int(section_pred_df_tmp.get("heuristic_fallback", False).fillna(False).sum()) if "heuristic_fallback" in section_pred_df_tmp else 0)
    print("missing-filled rows:", int(section_pred_df_tmp.get("missing_filled", False).fillna(False).sum()) if "missing_filled" in section_pred_df_tmp else 0)

Running LLM section labeling on 5 articles...


  0%|          | 0/5 [00:00<?, ?it/s]

Prediction(
    output_json='{"article_id":"PMC3002766","paragraph_sections":[\n{"pnum":0,"location_section":"title_abstract","content_sections":["title_abstract"]},\n{"pnum":1,"location_section":"title_abstract","content_sections":["title_abstract"]},\n{"pnum":2,"location_section":"title_abstract","content_sections":["title_abstract"]},\n{"pnum":3,"location_section":"title_abstract","content_sections":["title_abstract"]},\n{"pnum":4,"location_section":"title_abstract","content_sections":["title_abstract"]},\n{"pnum":5,"location_section":"title_abstract","content_sections":["title_abstract"]},\n{"pnum":6,"location_section":"title_abstract","content_sections":["title_abstract"]},\n{"pnum":7,"location_section":"title_abstract","content_sections":["title_abstract"]},\n{"pnum":8,"location_section":"title_abstract","content_sections":["title_abstract"]},\n{"pnum":9,"location_section":"title_abstract","content_sections":["title_abstract"]},\n{"pnum":10,"location_section":"title_abstract","co

In [35]:
def outputs_to_dataframe(section_outputs):
    rows = []
    for obj in section_outputs:
        rows.extend(obj.get("paragraph_sections", []))
    out_df = pd.DataFrame(rows)
    if len(out_df):
        out_df["content_sections_json"] = out_df["content_sections"].apply(json.dumps)
    return out_df

section_pred_df = outputs_to_dataframe(section_outputs)
section_pred_df.head()

,pnum,location_section,content_sections,missing_filled,article,pid,original_section_raw,original_location_section,content_sections_json
0,0,title_abstract,[title_abstract],False,PMC3002766,1,Title,title_abstract,"[""title_abstract""]"
1,1,title_abstract,[title_abstract],False,PMC3002766,2,Title,title_abstract,"[""title_abstract""]"
2,2,title_abstract,[title_abstract],False,PMC3002766,3,Abstract,title_abstract,"[""title_abstract""]"
3,3,title_abstract,[title_abstract],False,PMC3002766,4,Abstract,title_abstract,"[""title_abstract""]"
4,4,title_abstract,[title_abstract],False,PMC3002766,5,Abstract,title_abstract,"[""title_abstract""]"


In [36]:
# Sanity checks for leakage/copying
print("Rows:", len(section_pred_df))
if "heuristic_fallback" in section_pred_df.columns:
    print("heuristic_fallback rows:", int(section_pred_df["heuristic_fallback"].fillna(False).sum()))
if "oracle_location_copied" in section_pred_df.columns:
    print("oracle_location_copied rows:", int(section_pred_df["oracle_location_copied"].fillna(False).sum()))
if "missing_filled" in section_pred_df.columns:
    print("missing_filled rows:", int(section_pred_df["missing_filled"].fillna(False).sum()))

exact_loc_match = (section_pred_df["location_section"] == section_pred_df["original_location_section"]).mean()
primary_content = section_pred_df["content_sections"].apply(lambda xs: xs[0] if isinstance(xs, list) and xs else None)
exact_content_primary_match = (primary_content == section_pred_df["original_location_section"]).mean()
print("location exact match vs original:", round(float(exact_loc_match), 4))
print("content primary exact match vs original:", round(float(exact_content_primary_match), 4))


Rows: 380
missing_filled rows: 0
location exact match vs original: 0.8868
content primary exact match vs original: 0.8711


In [37]:
# Save compact JSON and flat CSV.
with open(SECTION_JSON_PATH, "w", encoding="utf-8") as f:
    json.dump(section_outputs, f, indent=2, ensure_ascii=False)

section_pred_df.to_csv(SECTION_CSV_PATH, index=False)
print("Saved:")
print(SECTION_JSON_PATH)
print(SECTION_CSV_PATH)

Saved:
../output_and_result/section_labeling/paragraph_section_labels_global_v1.json
../output_and_result/section_labeling/paragraph_section_labels_global_v1.csv


## 8. Evaluate section labeling against original dataset sections

Because the dataset contains the original structural section, the most direct evaluation is for `location_section`. For `content_sections`, the dataset does not provide independent multi-label content-based annotations, so we report a weak proxy: whether the original location section appears anywhere in the predicted content sections. A low score here is not necessarily wrong; it may indicate true content/location mismatch, which is exactly what this new stage is designed to capture.

In [38]:
def safe_div(a, b):
    return a / b if b else 0.0


def section_classification_report(y_true, y_pred, labels=CANONICAL_SECTIONS):
    rows = []
    for lab in labels:
        tp = int(((y_true == lab) & (y_pred == lab)).sum())
        fp = int(((y_true != lab) & (y_pred == lab)).sum())
        fn = int(((y_true == lab) & (y_pred != lab)).sum())
        precision = safe_div(tp, tp + fp)
        recall = safe_div(tp, tp + fn)
        f1 = safe_div(2 * precision * recall, precision + recall)
        support = int((y_true == lab).sum())
        rows.append({
            "section": lab,
            "precision": precision,
            "recall": recall,
            "f1": f1,
            "support": support,
            "tp": tp,
            "fp": fp,
            "fn": fn,
        })
    report = pd.DataFrame(rows)
    micro_acc = float((y_true == y_pred).mean()) if len(y_true) else 0.0
    macro_f1 = float(report["f1"].mean()) if len(report) else 0.0
    weighted_f1 = float(np.average(report["f1"], weights=np.maximum(report["support"], 1))) if len(report) else 0.0
    return report, {"accuracy_micro": micro_acc, "macro_f1": macro_f1, "weighted_f1": weighted_f1, "n": int(len(y_true))}


def evaluate_section_predictions(section_pred_df):
    df = section_pred_df.copy()
    df["content_contains_original_location"] = df.apply(
        lambda r: r["original_location_section"] in set(r.get("content_sections", []) or []), axis=1
    )
    df["content_primary_section"] = df["content_sections"].apply(lambda xs: xs[0] if isinstance(xs, list) and xs else "other_information")

    loc_report, loc_summary = section_classification_report(
        df["original_location_section"],
        df["location_section"],
    )
    content_primary_report, content_primary_summary = section_classification_report(
        df["original_location_section"],
        df["content_primary_section"],
    )
    content_contains_rate = float(df["content_contains_original_location"].mean()) if len(df) else 0.0

    summary = {
        **{f"location_{k}": v for k, v in loc_summary.items()},
        **{f"content_primary_vs_original_{k}": v for k, v in content_primary_summary.items()},
        "content_contains_original_location_rate": content_contains_rate,
        "n_articles": int(df[ARTICLE_ID_COL].nunique()) if ARTICLE_ID_COL in df.columns else int(df["article"].nunique()),
        "n_paragraphs": int(len(df)),
    }
    return df, loc_report, content_primary_report, summary

eval_df, location_report, content_primary_report, section_eval_summary = evaluate_section_predictions(section_pred_df)

print("Summary:")
print(json.dumps(section_eval_summary, indent=2))

print("\nLocation-section report:")
display(location_report)

print("\nContent-primary-section vs original-location proxy report:")
display(content_primary_report)

eval_df.to_csv(EVAL_CSV_PATH, index=False)
print("Saved evaluation rows to", EVAL_CSV_PATH)

Summary:
{
  "location_accuracy_micro": 0.8868421052631579,
  "location_macro_f1": 0.8503409309824576,
  "location_weighted_f1": 0.9113288181251834,
  "location_n": 380,
  "content_primary_vs_original_accuracy_micro": 0.8710526315789474,
  "content_primary_vs_original_macro_f1": 0.8447151271544037,
  "content_primary_vs_original_weighted_f1": 0.9011181808565735,
  "content_primary_vs_original_n": 380,
  "content_contains_original_location_rate": 0.8710526315789474,
  "n_articles": 5,
  "n_paragraphs": 380
}

Location-section report:


,section,precision,recall,f1,support,tp,fp,fn
0,title_abstract,0.969697,1.000000,0.984615,64,64,2,0
1,introduction,1.000000,1.000000,1.000000,23,23,0,0
2,methods,1.000000,0.972727,0.986175,110,107,0,3
3,results,1.000000,0.800000,0.888889,95,76,0,19
4,discussion,1.000000,0.736111,0.848000,72,53,0,19
5,other_information,0.254545,0.875000,0.394366,16,14,41,2



Content-primary-section vs original-location proxy report:


,section,precision,recall,f1,support,tp,fp,fn
0,title_abstract,1.000000,1.000000,1.000000,64,64,0,0
1,introduction,1.000000,1.000000,1.000000,23,23,0,0
2,methods,1.000000,0.954545,0.976744,110,105,0,5
3,results,1.000000,0.736842,0.848485,95,70,0,25
4,discussion,1.000000,0.736111,0.848000,72,53,0,19
5,other_information,0.246154,1.000000,0.395062,16,16,49,0


Saved evaluation rows to ../output_and_result/section_labeling/section_label_eval_v1.csv


In [24]:
# Confusion matrix for location_section vs original dataset section.
confusion = pd.crosstab(
    eval_df["original_location_section"],
    eval_df["location_section"],
    rownames=["original_location_section"],
    colnames=["pred_location_section"],
    dropna=False,
).reindex(index=CANONICAL_SECTIONS, columns=CANONICAL_SECTIONS, fill_value=0)
confusion

pred_location_section,title_abstract,introduction,methods,results,discussion,other_information
original_location_section,,,,,,
title_abstract,0,0,0,0,0,64
introduction,0,0,0,0,0,23
methods,0,0,0,0,0,110
results,0,0,0,0,0,95
discussion,0,0,0,0,0,72
other_information,0,0,0,0,0,16


## 9. Inspect content/location disagreement cases

These are the most important examples for the new logic: paragraphs where the structural location and content role differ.

In [ ]:
def attach_paragraph_text(eval_df, paragraph_df):
    keys = ["article", PARAGRAPH_ID_COL] if "article" in eval_df.columns else [ARTICLE_ID_COL, PARAGRAPH_ID_COL]
    text_df = paragraph_df[[ARTICLE_ID_COL, PARAGRAPH_ID_COL, PARAGRAPH_TEXT_COL]].rename(columns={ARTICLE_ID_COL: "article"})
    return eval_df.merge(text_df, on=["article", PARAGRAPH_ID_COL], how="left")

inspect_df = attach_paragraph_text(eval_df, paragraph_df)
inspect_df["content_differs_from_location"] = inspect_df.apply(
    lambda r: set(r["content_sections"]) != {r["location_section"]}, axis=1
)

disagreement_cols = [
    "article", PARAGRAPH_ID_COL, "pnum", "original_section_raw",
    "location_section", "content_sections", PARAGRAPH_TEXT_COL,
]

inspect_df.loc[inspect_df["content_differs_from_location"], disagreement_cols].head(20)

In [ ]:
# Counts of content-section combinations.
combo_counts = (
    eval_df["content_sections"]
    .apply(lambda xs: ",".join(xs) if isinstance(xs, list) else str(xs))
    .value_counts()
    .to_frame("n")
)
combo_counts.head(30)

## 10. Minimal interface for the next extraction stage

The next version can consume `section_pred_df` / `paragraph_section_labels_global_v1.json` and route paragraphs to section-specific extraction modules. The important contract is:

- `location_section`: single canonical section.
- `content_sections`: one or more canonical sections.
- Use `content_sections` for extraction routing.
- Keep `location_section` as a prior/feature, not as a hard constraint.

In [ ]:
def get_paragraphs_for_section(section_pred_df, target_section):
    """Return rows whose content-based section labels include target_section."""
    target_section = normalize_section_name(target_section)
    return section_pred_df[
        section_pred_df["content_sections"].apply(lambda xs: target_section in set(xs or []))
    ].copy()

# Example for next-stage extraction routing:
methods_like = get_paragraphs_for_section(section_pred_df, "methods")
results_like = get_paragraphs_for_section(section_pred_df, "results")
print("methods-like paragraphs:", len(methods_like))
print("results-like paragraphs:", len(results_like))